------------------
```markdown
# Copyright © 2025 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amendments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com
```
------------------------------
❗❗❗ **IMPORTANT**❗❗❗ **Create a copy of this notebook**

In order to work with this Google Colab you need to create a copy of it. Please **DO NOT** provide your answers here. Instead, work on the copy version. To make a copy:

**Click on: File -> save a copy in drive**

Have you successfully created the copy? if yes, there must be a new tab opened in your browser. Now move to the copy and start from there!

----------------------------------------------


# Bayesian Hierarchical Model

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand the concept of hierarchical modeling and partial pooling
2. Build hierarchical models with varying intercepts and slopes
3. Interpret hyperparameters and shrinkage effects
4. Use posterior predictive checks for model validation
5. Compare complete pooling, no pooling, and partial pooling approaches

In [ ]:
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
az.style.use("arviz-darkgrid")

## Why Hierarchical Models?

**Our example: Student Math Performance Across Schools**
- **Data:** Math scores from students across 10 Italian middle schools
- **Structure:** Students nested within schools
- **Question:** How does baseline math performance (math_t1) predict final math performance (math_t2)?

**Three approaches to modeling grouped data:**

1. **Complete Pooling**: Ignore schools entirely
   - One regression line for all students
   - Assumes all schools are identical
   - High bias, low variance

2. **No Pooling**: Separate regression per school
   - Independent estimates for each school
   - Ignores similarities between schools
   - Low bias, high variance (especially for small schools)

3. **Partial Pooling (Hierarchical)**: Schools share information
   - Best of both worlds!
   - Schools borrow strength from each other
   - Automatic regularization via shrinkage

| **Model** | **Equation** |
|-----------|:------------:|
| **Complete Pooling** | $$Y_i = \alpha + \beta X_i + \epsilon_i$$ |
| **No Pooling** | $$Y_i = \alpha_{j[i]} + \beta_{j[i]} X_i + \epsilon_i$$ |
| **Partial Pooling (Random Intercept)** | $$Y_i = \alpha_{j[i]} + \beta X_i + \epsilon_i$$ |
| **Partial Pooling (Random Slope)** | $$Y_i = \alpha + \beta_{j[i]} X_i + \epsilon_i$$ |
| **Partial Pooling (Both)** | $$Y_i = \alpha_{j[i]} + \beta_{j[i]} X_i + \epsilon_i$$ |

### Dataset: Italian Middle School Students

**Variables:**
- `math_t1`: Math score at time 1 (baseline, 0-10 scale)
- `math_t2`: Math score at time 2 (outcome, 0-10 scale)
- `school_id`: School identifier

**Important manipulation:** We'll artificially reduce the sample size for 2 schools to demonstrate shrinkage effects!

In [ ]:
# Load data
df_full = pd.read_csv("https://drive.google.com/uc?id=1znf0o6-4IqQ7EtvRCZxVYtjUjBNCmZuP")

# Select relevant columns
df = df_full[['school_id', 'student_id', 'math_t1', 'math_t2']].copy()

# Create school index
df['school_idx'], school_names = pd.factorize(df['school_id'])
n_schools = df['school_idx'].nunique()

print("Original sample sizes by school:")
print(df.groupby('school_id').size())

# ARTIFICIALLY REDUCE sample size for schools 0 and 1
# This will demonstrate shrinkage in hierarchical models
np.random.seed(42)

# School 0: Keep only 10 students (from ~150)
school_0_students = df[df['school_idx'] == 0]['student_id'].values
keep_0 = np.random.choice(school_0_students, size=12, replace=False)

# School 1: Keep only 20 students (from ~150)
school_1_students = df[df['school_idx'] == 3]['student_id'].values
keep_1 = np.random.choice(school_1_students, size=11, replace=False)

# Filter dataset
df = df[
    (df['school_idx'] == 0) & (df['student_id'].isin(keep_0)) |
    (df['school_idx'] == 3) & (df['student_id'].isin(keep_1)) |
    (~df['school_idx'].isin([0, 3]))
].copy()

# Re-index schools
df['school_idx'], school_names = pd.factorize(df['school_id'])
n_schools = df['school_idx'].nunique()

print("\n" + "="*70)
print("MANIPULATED sample sizes (for teaching purposes):")
print("="*70)
sample_sizes = df.groupby('school_id').size()
print(sample_sizes)
print(f"\nTotal students: {len(df)}")
print(f"Number of schools: {n_schools}")
print(f"\n⚠️  Schools {school_names[0]} and {school_names[1]} have artificially small samples!")
print(f"   This will help us see shrinkage effects in hierarchical models.")

Let's visualize the relationship between baseline (math_t1) and final (math_t2) math scores.

In [ ]:
# Visualize data
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Overall relationship
axes[0, 0].scatter(df['math_t1'], df['math_t2'], alpha=0.5, s=30, color='steelblue')
axes[0, 0].plot([0, 10], [0, 10], 'r--', linewidth=2, label='Perfect correlation')
axes[0, 0].set_xlabel('Math T1 (Baseline)', fontsize=11)
axes[0, 0].set_ylabel('Math T2 (Final)', fontsize=11)
axes[0, 0].set_title('Overall: Math T1 vs T2', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. By school (colored)
for j in range(n_schools):
    school_data = df[df['school_idx'] == j]
    axes[0, 1].scatter(school_data['math_t1'], school_data['math_t2'],
                       alpha=0.6, s=40, label=f'School {j}')
axes[0, 1].set_xlabel('Math T1 (Baseline)', fontsize=11)
axes[0, 1].set_ylabel('Math T2 (Final)', fontsize=11)
axes[0, 1].set_title('By School: Math T1 vs T2', fontsize=12, fontweight='bold')
axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# 3. Sample sizes
sample_sizes = df.groupby('school_id').size()
colors = ['red' if n < 50 else 'steelblue' for n in sample_sizes.values]
axes[1, 0].bar(range(len(sample_sizes)), sample_sizes.values,
               color=colors, edgecolor='black')
axes[1, 0].set_xlabel('School', fontsize=11)
axes[1, 0].set_ylabel('Number of Students', fontsize=11)
axes[1, 0].set_title('Sample Size by School (Red = Small)', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(range(n_schools))
axes[1, 0].set_xticklabels([f'S{i}' for i in range(n_schools)])
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. School-level correlations
school_corrs = df.groupby('school_idx')[['math_t1', 'math_t2']].corr().iloc[0::2, 1].values
colors = ['red' if sample_sizes.values[i] < 50 else 'steelblue' for i in range(n_schools)]
axes[1, 1].bar(range(n_schools), school_corrs, color=colors, edgecolor='black')
axes[1, 1].set_xlabel('School', fontsize=11)
axes[1, 1].set_ylabel('Correlation (Math T1 vs T2)', fontsize=11)
axes[1, 1].set_title('Within-School Correlations', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(range(n_schools))
axes[1, 1].set_xticklabels([f'S{i}' for i in range(n_schools)])
axes[1, 1].axhline(df[['math_t1', 'math_t2']].corr().iloc[0, 1],
                   color='green', linestyle='--', linewidth=2, label='Overall')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print(f"1. Overall correlation: {df[['math_t1', 'math_t2']].corr().iloc[0,1]:.3f}")
print(f"2. Schools 0 and 1 have very small samples (n={sample_sizes.values[0]}, {sample_sizes.values[1]})")
print(f"3. Other schools have balanced samples (~140-160 students)")
print(f"4. Perfect setup to demonstrate shrinkage!")

## Model 1: Complete Pooling

Ignore schools entirely - fit one regression for all students.

**Model:**
$$\text{math_t2}_i = \alpha + \beta \cdot \text{math_t1}_i + \epsilon_i$$

**Problem:** Assumes all schools are identical!

In [ ]:
# Simple linear regression (complete pooling)
from sklearn.linear_model import LinearRegression

# Fit OLS
X = df[['math_t1']].values
y = df['math_t2'].values

lr = LinearRegression()
lr.fit(X, y)

print("Complete Pooling (OLS):")
print(f"  Intercept (α): {lr.intercept_:.3f}")
print(f"  Slope (β): {lr.coef_[0]:.3f}")
print(f"  R²: {lr.score(X, y):.3f}")
print(f"\nInterpretation: One line fits all {len(df)} students, ignoring schools")

## Model 2: No Pooling

Separate regression for each school - no information sharing.

**Model:**
$$\text{math\_t2}_i = \alpha_{j[i]} + \beta_{j[i]} \cdot \text{math\_t1}_i + \epsilon_i$$

**Problem:** Small schools get unreliable estimates!

In [ ]:
# Separate regression per school (no pooling)
no_pool_results = []

print("No Pooling (Separate OLS per school):")
print("="*70)
for j in range(n_schools):
    school_data = df[df['school_idx'] == j]
    X_school = school_data[['math_t1']].values
    y_school = school_data['math_t2'].values

    lr_school = LinearRegression()
    lr_school.fit(X_school, y_school)

    no_pool_results.append({
        'school': j,
        'n': len(school_data),
        'alpha': lr_school.intercept_,
        'beta': lr_school.coef_[0]
    })

    print(f"School {j}: n={len(school_data):3d}, α={lr_school.intercept_:6.3f}, β={lr_school.coef_[0]:6.3f}")

print(f"\nProblem: Schools 0 and 1 have very few students!")
print(f"  → Their estimates are highly uncertain")
print(f"  → No borrowing of strength from other schools")

## Model 3: Partial Pooling (Hierarchical)

Schools share information through hyperparameters.

**Random Intercepts Model:**
$$
\begin{aligned}
\text{math\_t2}_i &\sim \mathcal{N}(\mu_i, \sigma^2) \\
\mu_i &= \alpha_{j[i]} + \beta \cdot \text{math\_t1}_i \\
\alpha_j &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\
\mu_\alpha &\sim \mathcal{N}(0, 5) \\
\sigma_\alpha &\sim \text{HalfCauchy}(2) \\
\beta &\sim \mathcal{N}(1, 1) \\
\sigma &\sim \text{HalfCauchy}(2)
\end{aligned}
$$

**Key:** School-specific intercepts, common slope, sharing through $(\mu_\alpha, \sigma_\alpha)$

In [ ]:
# Partial pooling: Random intercepts
school_idx = df['school_idx'].values
math_t1_obs = df['math_t1'].values
math_t2_obs = df['math_t2'].values

with pm.Model() as partial_pool_model:
    # Hyperpriors
    mu_alpha = pm.Normal('mu_alpha', mu=0, sigma=5)
    sigma_alpha = pm.HalfCauchy('sigma_alpha', 2)
    # Common slope
    beta = pm.Normal('beta', mu=0, sigma=1)
    # School-specific intercepts
    alpha_school = pm.Normal('alpha_school', mu=mu_alpha, sigma=sigma_alpha,
                             shape=n_schools)
    # alpha_school = mu_alpha + sigma_alpha*pm.Normal('alpha_school', mu=0, sigma=1, shape=n_schools)

    # Within-school variation
    sigma = pm.HalfCauchy('sigma', 2)

    # Linear model
    mu = alpha_school[school_idx] + beta * math_t1_obs

    # Likelihood
    math_t2_like = pm.Normal('math_t2_like', mu=mu, sigma=sigma,
                             observed=math_t2_obs)

    # Sample
    trace_partial = pm.sample(2000, tune=1000, target_accept=0.98, chains=4,
                              cores=4,
                              random_seed=42, return_inferencedata=True)

print("\nPartial Pooling Model:")
print(f"- {n_schools} school-specific intercepts")
print(f"- 1 common slope")
print(f"- Information sharing through hyperparameters")

### Visualize Shrinkage Effect

Compare no pooling vs. partial pooling estimates.

In [ ]:
# Extract posterior means
alpha_partial = trace_partial.posterior['alpha_school'].mean(dim=['chain', 'draw']).values
alpha_partial_hdi = az.hdi(trace_partial, var_names=['alpha_school'], hdi_prob=0.94)['alpha_school'].values
beta_partial = trace_partial.posterior['beta'].mean().values

# Get no pooling estimates
alpha_no_pool = np.array([r['alpha'] for r in no_pool_results])
sample_sizes = np.array([r['n'] for r in no_pool_results])

# Global mean
mu_alpha_post = trace_partial.posterior['mu_alpha'].mean().values

# Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Intercepts: No pooling vs Partial pooling
axes[0, 0].scatter(range(n_schools), alpha_no_pool, s=100, color='red',
                   marker='x', linewidth=2, zorder=4, label='No pooling')
axes[0, 0].scatter(range(n_schools), alpha_partial, s=100, color='steelblue',
                   edgecolors='black', linewidth=1, zorder=3, label='Partial pooling')
axes[0, 0].vlines(range(n_schools), alpha_partial_hdi[:, 0], alpha_partial_hdi[:, 1],
                  colors='steelblue', linewidth=2, alpha=0.6)
axes[0, 0].axhline(mu_alpha_post, color='green', linestyle='--',
                   linewidth=2, label='Global mean (μ_α)')
axes[0, 0].set_xlabel('School', fontsize=11)
axes[0, 0].set_ylabel('Intercept (α)', fontsize=11)
axes[0, 0].set_title('No Pooling vs Partial Pooling', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Shrinkage magnitude
shrinkage = np.abs(alpha_no_pool - alpha_partial)
colors = ['red' if n < 50 else 'steelblue' for n in sample_sizes]
axes[0, 1].bar(range(n_schools), shrinkage, color=colors, edgecolor='black')
axes[0, 1].set_xlabel('School', fontsize=11)
axes[0, 1].set_ylabel('Shrinkage |α_no_pool - α_partial|', fontsize=11)
axes[0, 1].set_title('Shrinkage by School (Red = Small n)', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(range(n_schools))
axes[0, 1].set_xticklabels([f'S{i}' for i in range(n_schools)])
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Shrinkage vs sample size
axes[1, 0].scatter(sample_sizes, shrinkage, s=100, color='purple',
                   edgecolors='black', linewidth=1, alpha=0.7)
for i in range(n_schools):
    axes[1, 0].text(sample_sizes[i], shrinkage[i], f' S{i}',
                    fontsize=9, va='center')
axes[1, 0].set_xlabel('Sample Size', fontsize=11)
axes[1, 0].set_ylabel('Shrinkage', fontsize=11)
axes[1, 0].set_title('Shrinkage vs Sample Size', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Hyperparameters
az.plot_posterior(trace_partial, var_names=['mu_alpha', 'sigma_alpha', 'beta'],
                  ax=axes[1, 1:], hdi_prob=0.94)
axes[1, 1].set_title('Hyperparameters', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nShrinkage Analysis:")
print("="*70)
for i in range(n_schools):
    print(f"School {i}: n={sample_sizes[i]:3d}, shrinkage={shrinkage[i]:.3f}")

print(f"\n✓ Small schools (0, 1) show MUCH more shrinkage!")
print(f"✓ Large schools stay close to their sample estimates")
print(f"✓ This is automatic partial pooling at work!")

## Posterior Predictive Checks (PPC)

Validate the partial pooling model by checking:
1. **Overall distribution** - Does model capture data distribution?
2. **Residuals** - Are residuals well-behaved (centered at 0, constant variance)?
3. **Test statistics** - Can model reproduce key summary statistics?

In [ ]:
# Generate posterior predictive samples
with partial_pool_model:
    ppc = pm.sample_posterior_predictive(trace_partial, random_seed=42)

ppc_samples = ppc.posterior_predictive['math_t2_like'].values
observed = math_t2_obs

print(f"Generated {ppc_samples.shape[1]} posterior predictive samples")
print(f"Each sample contains {ppc_samples.shape[2]} predictions")

### Overall Distribution

In [ ]:
# Check 1: Overall distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
axes[0].hist(observed, bins=30, alpha=0.6, color='blue',
             edgecolor='black', label='Observed data', density=True)
for i in range(min(10, ppc_samples.shape[1])):
    axes[0].hist(ppc_samples[0, i, :], bins=30, alpha=0.1,
                 color='red', density=True, label=f'Predicted (10 samples - each {ppc_samples.shape[2]} datapoints)' if i==0 else [])
axes[0].set_xlabel('Math T2', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('PPC: Overall Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# QQ plot
obs_sorted = np.sort(observed)
pred_mean = ppc_samples.mean(axis=(0, 1))
pred_sorted = np.sort(pred_mean)

axes[1].scatter(obs_sorted, pred_sorted, alpha=0.5, s=20, color='purple')
axes[1].plot([observed.min(), observed.max()],
             [observed.min(), observed.max()],
             'r--', linewidth=2, label='Perfect fit')
axes[1].set_xlabel('Observed Quantiles', fontsize=11)
axes[1].set_ylabel('Predicted Quantiles', fontsize=11)
axes[1].set_title('Q-Q Plot', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Model captures overall distribution well")

### Residuals

In [ ]:
# Check 2: Residuals
# Get posterior mean predictions
posterior_mean_pred = ppc_samples.mean(axis=(0, 1))
residuals = observed - posterior_mean_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2a. Residuals vs fitted
axes[0, 0].scatter(posterior_mean_pred, residuals, alpha=0.5, s=30, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values', fontsize=11)
axes[0, 0].set_ylabel('Residuals', fontsize=11)
axes[0, 0].set_title('Residuals vs Fitted', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2b. Residuals vs predictor
axes[0, 1].scatter(math_t1_obs, residuals, alpha=0.5, s=30, color='coral')
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Math T1 (Predictor)', fontsize=11)
axes[0, 1].set_ylabel('Residuals', fontsize=11)
axes[0, 1].set_title('Residuals vs Predictor', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 2c. Histogram of residuals
axes[1, 0].hist(residuals, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Residuals', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 2d. QQ plot of residuals
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Normal Q-Q Plot of Residuals', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nResidual Diagnostics:")
print(f"  Mean: {residuals.mean():.4f} (should be ~0)")
print(f"  Std: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}, Max: {residuals.max():.4f}")
print(f"\n✓ Residuals centered at 0")
print(f"✓ No strong patterns in residual plots")
print(f"✓ Approximately normal distribution")

### Test Statistics

In [ ]:
# Check 3: Test statistics
# Can the model reproduce key summary statistics?

def calc_test_stats(data):
    """Calculate test statistics"""
    return {
        'mean': np.mean(data),
        'std': np.std(data),
        'median': np.median(data),
        'q25': np.percentile(data, 25),
        'q75': np.percentile(data, 75)
    }

# Observed statistics
obs_stats = calc_test_stats(observed)

# Predicted statistics for each PPC sample
ppc_stats = {
    'mean': [],
    'std': [],
    'median': [],
    'q25': [],
    'q75': []
}

for i in range(ppc_samples.shape[1]):
    sample_stats = calc_test_stats(ppc_samples[0, i, :])
    for key in ppc_stats:
        ppc_stats[key].append(sample_stats[key])

# Plot
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

stat_names = list(obs_stats.keys())
for idx, stat in enumerate(stat_names):
    axes[idx].hist(ppc_stats[stat], bins=30, alpha=0.7, color='lightblue',
                   edgecolor='black', label='Predicted')
    axes[idx].axvline(obs_stats[stat], color='red', linewidth=2,
                      linestyle='--', label='Observed')
    axes[idx].set_xlabel(stat.upper(), fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    axes[idx].set_title(f'PPC: {stat.upper()}', fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print p-values (Bayesian p-value: proportion of PPC samples more extreme than observed)
print("\nBayesian p-values (proportion of PPC samples more extreme than observed):")
print("="*70)
for stat in stat_names:
    ppc_array = np.array(ppc_stats[stat])
    p_value = np.mean(ppc_array > obs_stats[stat]) if obs_stats[stat] < np.median(ppc_array) else np.mean(ppc_array < obs_stats[stat])
    p_value = 2 * min(p_value, 1 - p_value)  # Two-tailed
    print(f"  {stat.upper():8s}: obs={obs_stats[stat]:7.3f}, p-value={p_value:.3f}")

print(f"\n✓ Model reproduces all key statistics well (p-values > 0.05)")
print(f"✓ Observed values fall within predicted distributions")


## Exercise 1: Random Slopes Model

Extend the hierarchical model to allow slopes to vary by school:

**Model:**
$$
\begin{aligned}
\text{math_t2}_i &\sim \mathcal{N}(\mu_i, \sigma^2) \\
\mu_i &= \alpha_{j[i]} + \beta_{j[i]} \cdot \text{math_t1}_i \\
\alpha_j &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\
\beta_j &\sim \mathcal{N}(\mu_\beta, \sigma_\beta^2)
\end{aligned}
$$

**Tasks:**
1. Build the model with both random intercepts and random slopes
2. Compare school-specific slopes
3. Visualize: Do small schools (0, 1) show shrinkage in slopes too?
4. Interpret: Which schools have stronger/weaker math_t1 → math_t2 relationships?

## Exercise 2: Model Comparison (WAIC and LOO-CV)

Compare the three models using information criteria:

**Models to compare:**
1. Complete pooling (needs to be fit with PyMC for comparison)
2. Partial pooling - Random intercepts only
3. Partial pooling - Random intercepts + Random slopes

**Tasks:**
1. Build complete pooling model in PyMC
2. Compute WAIC for all three models: `az.waic(trace)`
3. Compute LOO for all three models: `az.loo(trace)`
4. Compare models: `az.compare({'model1': trace1, 'model2': trace2, ...}, ic='waic')`
5. Visualize comparison: `az.plot_compare(comparison)`
6. **Questions:**
   - Which model fits best?
   - Is the complexity of random slopes justified?
   - What is the effective number of parameters for each model?


## Exercise 3: Add Covariates

Extend the random intercept model to include student-level covariates:

**Variables available:**
- `sex` (1=female, 2=male)
- `nationality` (1=Italian, 2=foreign)

**Model:**
$$\mu_i = \alpha_{j[i]} + \beta \cdot \text{math\_t1}_i + \gamma_1 \cdot \text{sex}_i + \gamma_2 \cdot \text{nationality}_i$$

**Tasks:**
1. Load sex and nationality from the original dataset
2. Build the extended model
3. Interpret: Do girls/boys or Italian/foreign students perform differently?
4. Check: Does adding covariates change school-level shrinkage patterns?

**Congratulations! You have finished the Notebook! Great Job!** 🤗🙌👍👏💪

<!--
# Copyright © 2025 Meysam Goodarzi
This notebook is licensed under CC BY-NC 4.0 with the following amendments:
- Individuals may use, share, and adapt this material for non-commercial purposes with attribution.
- Institutions/Companies must obtain written consent to use this material, except for nonprofits.
- Commercial use is prohibited without permission.  
Contact: analytica@meysam-goodarzi.com.
-->